# Verify distributed-training shape invariants

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thehalleyyoung/tensorguard/blob/main/examples/tutorials/07_distributed.ipynb)

FSDP and DeepSpeed change physical parameter layout but must preserve the logical tensor shapes the forward pass expects. TensorGuard checks those sharding contracts without launching a distributed job.

In [ ]:
%pip install -q "git+https://github.com/thehalleyyoung/tensorguard.git"

In [ ]:
from src.distributed_verification import FSDPConfig, verify_distributed

source = '''
import torch.nn as nn
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(256, 128)
        self.fc2 = nn.Linear(128, 10)
    def forward(self, x):
        return self.fc2(self.fc1(x))
'''
result = verify_distributed(source, input_shapes={'x': ('batch', 256)},
                            fsdp_config=FSDPConfig(world_size=4))
print(result.pretty())
assert result.safe
assert result.fsdp_result is not None and result.fsdp_result.safe